In [15]:

from phm_america_2024.common.path_service_common import find_project_root
import duckdb
con = duckdb.connect()
import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt
from IPython.display import display
import joblib


In [18]:

# 1. Localiza la raíz del proyecto automáticamente
root = find_project_root()

# 2. Define la ruta hacia tu carpeta de salida (la base de tus 'runs')
# Asumimos que quieres la última ejecución o una específica
run_id = "20260607_133914" # O el ID que estés analizando
base_run_dir = root / "outputs" / "runs" / "regression" / "phm2024" / run_id

In [19]:
######################################### SELECTIONS #########################################

In [20]:

path_x_y_train_3_1 = base_run_dir / "phase3_data_preparation" / ("3.1.selection.selected_regression_train.parquet")

query_conteos = f"""
SELECT 'x_y_train_3_1' as dataset, count(*) as total_registros FROM parquet_scan('{path_x_y_train_3_1}')
"""
print(duckdb.query(query_conteos).to_df())
print("----------------------------------------------------")
query_schema_train = f"DESCRIBE SELECT * FROM parquet_scan('{path_x_y_train_3_1}')"
print("--- ESTRUCTURA DATASET DE x_y_train_3_1 ---")
print(duckdb.query(query_schema_train).to_df()[['column_name', 'column_type']])
print("----------------------------------------------------")
query = f"SELECT * FROM '{path_x_y_train_3_1}' LIMIT 5"
df_duck = con.sql(query).df()
display(df_duck)

         dataset  total_registros
0  x_y_train_3_1             6994
----------------------------------------------------
--- ESTRUCTURA DATASET DE x_y_train_3_1 ---
    column_name column_type
0           oat      DOUBLE
1           mgt      DOUBLE
2            pa      DOUBLE
3           ias      DOUBLE
4            np      DOUBLE
5            ng      DOUBLE
6    trq_margin      DOUBLE
7  trq_measured      DOUBLE
----------------------------------------------------


,oat,mgt,pa,ias,np,ng,trq_margin,trq_measured
0,11.250000,593.6000,126.18720,113.93750,100.13000,94.18000,11.579641,77.70000
1,-10.250000,648.9000,522.12240,127.31250,95.64000,100.09000,-16.800329,88.00000
2,10.168530,577.2656,-81.72695,61.47461,99.77149,93.05847,7.471748,70.76563
3,21.000000,586.2000,250.85040,101.62500,99.74000,91.91000,5.075661,60.70000
4,9.252266,542.0938,538.20880,113.04490,99.58594,90.77332,-70.763798,15.76538


In [21]:
######################################### CLEANING #########################################

In [22]:
# El paso 3.2 aplicó una técnica llamada Clipping (o Winsorization) a la variable
# objetivo (trq_margin)
path_x_y_train_3_2 = base_run_dir / "phase3_data_preparation" / ("3.2.cleaning.cleaned_regression_train.parquet")

query_conteos = f"""
SELECT 'x_y_train_3_2' as dataset, count(*) as total_registros FROM parquet_scan('{path_x_y_train_3_2}')
"""
print(duckdb.query(query_conteos).to_df())
print("----------------------------------------------------")
query_schema_train = f"DESCRIBE SELECT * FROM parquet_scan('{path_x_y_train_3_2}')"
print("--- ESTRUCTURA DATASET DE x_y_train_3_2 ---")
print(duckdb.query(query_schema_train).to_df()[['column_name', 'column_type']])
print("----------------------------------------------------")
query = f"SELECT * FROM '{path_x_y_train_3_2}' LIMIT 5"
df_duck = con.sql(query).df()
display(df_duck)

         dataset  total_registros
0  x_y_train_3_2             6994
----------------------------------------------------
--- ESTRUCTURA DATASET DE x_y_train_3_2 ---
    column_name column_type
0           oat      DOUBLE
1           mgt      DOUBLE
2            pa      DOUBLE
3           ias      DOUBLE
4            np      DOUBLE
5            ng      DOUBLE
6    trq_margin      DOUBLE
7  trq_measured      DOUBLE
----------------------------------------------------


,oat,mgt,pa,ias,np,ng,trq_margin,trq_measured
0,11.250000,593.6000,126.18720,113.93750,100.13000,94.18000,11.579641,77.70000
1,-10.250000,648.9000,522.12240,127.31250,95.64000,100.09000,-16.800329,88.00000
2,10.168530,577.2656,-81.72695,61.47461,99.77149,93.05847,7.471748,70.76563
3,21.000000,586.2000,250.85040,101.62500,99.74000,91.91000,5.075661,60.70000
4,9.252266,542.0938,538.20880,113.04490,99.58594,90.77332,-70.763798,15.76538


In [23]:

# Ejecutamos una query que une la información del paso 3.1 y el 3.2
query_clipping_audit = f"""
SELECT
    '3.1 - Antes de Limpieza (Original)' AS etapa,
    MIN(trq_margin) AS trq_margin_minimo,
    MAX(trq_margin) AS trq_margin_maximo,
    COUNT(*) AS total_filas
FROM parquet_scan('{path_x_y_train_3_1}')

UNION ALL

SELECT
    '3.2 - Después de Limpieza (Clipped)' AS etapa,
    MIN(trq_margin) AS trq_margin_minimo,
    MAX(trq_margin) AS trq_margin_maximo,
    COUNT(*) AS total_filas
FROM parquet_scan('{path_x_y_train_3_2}')
"""

print(f"\n{'='*70}")
print("📊 AUDITORÍA DE CLIPPING: Variable 'trq_margin' (Paso 3.1 vs 3.2)")
print(f"{'='*70}")

# Mostramos el resultado
df_audit = duckdb.query(query_clipping_audit).to_df()
display(df_audit)


📊 AUDITORÍA DE CLIPPING: Variable 'trq_margin' (Paso 3.1 vs 3.2)


,etapa,trq_margin_minimo,trq_margin_maximo,total_filas
0,3.1 - Antes de Limpieza (Original),-74.523751,29.896175,6994
1,3.2 - Después de Limpieza (Clipped),-72.729422,17.723187,6994


In [24]:

# Agrupamos los paths de 3.1 y 3.2
datasets_iniciales = {
    "3.1 - SELECCIÓN": path_x_y_train_3_1,
    "3.2 - LIMPIEZA": path_x_y_train_3_2
}

for nombre, ruta in datasets_iniciales.items():
    print(f"\n{'='*60}")
    print(f"🔍 Auditando Dataset: {nombre}")
    print(f"📁 Ruta: {ruta}")
    print(f"{'='*60}")

    try:
        # Cargar todo el dataset
        query = f"SELECT * FROM parquet_scan('{ruta}')"
        df = duckdb.query(query).to_df()

        # Aislar numéricas
        df_num = df.select_dtypes(include=[np.number])

        # Diagnóstico
        diagnostico = pd.DataFrame({
            'Total Nulos (NaN)': df_num.isna().sum(),
            'Infinitos (+inf)': np.isposinf(df_num).sum(),
            'Infinitos (-inf)': np.isneginf(df_num).sum(),
            'Valor Máximo': df_num.max(),
            'Valor Mínimo': df_num.min()
        })

        # Filtrar rotas
        columnas_rotas = diagnostico[
            (diagnostico['Infinitos (+inf)'] > 0) |
            (diagnostico['Infinitos (-inf)'] > 0) |
            (diagnostico['Total Nulos (NaN)'] > 0)
            ]

        if columnas_rotas.empty:
            print(f"✅ ¡Excelente! El dataset {nombre} está completamente LIMPIO de NaNs e Infinitos.\n")

            # Verificamos si specific_power_index existe aquí
            if 'specific_power_index' in df.columns:
                print("ℹ️ Nota: 'specific_power_index' SÍ existe en esta etapa.")
            else:
                print("ℹ️ Nota: 'specific_power_index' NO existe en esta etapa (se calcula después).")
        else:
            print(f"⚠️ ¡ALERTA! Se encontraron datos corruptos en {nombre}:")
            display(columnas_rotas)

    except Exception as e:
        print(f"❌ Error al auditar {nombre}: {e}\n")


🔍 Auditando Dataset: 3.1 - SELECCIÓN
📁 Ruta: K:\00_Code\Manutenzione\Project_MPPR-AI_B_PHM_America_2024\outputs\runs\regression\phm2024\20260607_133914\phase3_data_preparation\3.1.selection.selected_regression_train.parquet
✅ ¡Excelente! El dataset 3.1 - SELECCIÓN está completamente LIMPIO de NaNs e Infinitos.

ℹ️ Nota: 'specific_power_index' NO existe en esta etapa (se calcula después).

🔍 Auditando Dataset: 3.2 - LIMPIEZA
📁 Ruta: K:\00_Code\Manutenzione\Project_MPPR-AI_B_PHM_America_2024\outputs\runs\regression\phm2024\20260607_133914\phase3_data_preparation\3.2.cleaning.cleaned_regression_train.parquet
✅ ¡Excelente! El dataset 3.2 - LIMPIEZA está completamente LIMPIO de NaNs e Infinitos.

ℹ️ Nota: 'specific_power_index' NO existe en esta etapa (se calcula después).


In [25]:
######################################### TRANSFORMATIONS #########################################

In [26]:
path_x_y_train_3_3 = base_run_dir / "phase3_data_preparation" / ("3.3.transformation.transformed_regression_train.parquet")
query_conteos = f"""
SELECT 'x_y_train_3_3' as dataset, count(*) as total_registros FROM parquet_scan('{path_x_y_train_3_3}')
"""
print(duckdb.query(query_conteos).to_df())
print("----------------------------------------------------")
query_schema_train = f"DESCRIBE SELECT * FROM parquet_scan('{path_x_y_train_3_3}')"
print("--- ESTRUCTURA DATASET DE x_y_train_3_3 ---")
print(duckdb.query(query_schema_train).to_df()[['column_name', 'column_type']])
print("----------------------------------------------------")
query = f"SELECT * FROM '{path_x_y_train_3_3}' LIMIT 5"
df_duck = con.sql(query).df()
display(df_duck)

         dataset  total_registros
0  x_y_train_3_3             6994
----------------------------------------------------
--- ESTRUCTURA DATASET DE x_y_train_3_3 ---
            column_name column_type
0                   oat      DOUBLE
1                   mgt      DOUBLE
2                    pa      DOUBLE
3                   ias      DOUBLE
4                    np      DOUBLE
5                    ng      DOUBLE
6            trq_margin      DOUBLE
7          trq_measured      DOUBLE
8         mgt_oat_ratio      DOUBLE
9  specific_power_index      DOUBLE
----------------------------------------------------


,oat,mgt,pa,ias,np,ng,trq_margin,trq_measured,mgt_oat_ratio,specific_power_index
0,-0.202533,0.064560,-0.465015,0.429291,0.077124,-0.080735,0.666394,0.641576,2.087201,0.003738
1,-2.137852,0.979980,0.404754,0.657252,-0.858781,0.688854,-1.273859,1.186549,2.468239,0.003635
2,-0.299882,-0.205836,-0.921750,-0.464876,0.002395,-0.226779,0.385550,0.274678,2.037514,0.003784
3,0.675111,-0.057938,-0.191162,0.219439,-0.004169,-0.376331,0.221736,-0.257895,1.992861,0.003689
4,-0.382359,-0.788060,0.440092,0.414077,-0.036281,-0.524347,-4.963179,-2.635388,1.919580,0.003885


In [28]:


print(f"\n{'='*60}")
print(f"🔍 Auditando Dataset: TRAIN (Step 3.3 - Transformación)")
print(f"📁 Ruta: {path_x_y_train_3_3}")
print(f"{'='*60}")

try:
    # Cargar el dataset completo de 3.3 usando DuckDB
    query_3_3 = f"SELECT * FROM parquet_scan('{path_x_y_train_3_3}')"
    df_3_3 = duckdb.query(query_3_3).to_df()

    # Aislar columnas numéricas
    df_num_3_3 = df_3_3.select_dtypes(include=[np.number])

    # Diagnóstico de NaNs e Infinitos
    diagnostico_3_3 = pd.DataFrame({
        'Total Nulos (NaN)': df_num_3_3.isna().sum(),
        'Infinitos (+inf)': np.isposinf(df_num_3_3).sum(),
        'Infinitos (-inf)': np.isneginf(df_num_3_3).sum(),
        'Valor Máximo': df_num_3_3.max(),
        'Valor Mínimo': df_num_3_3.min()
    })

    # Filtrar solo las columnas problemáticas
    columnas_rotas_3_3 = diagnostico_3_3[
        (diagnostico_3_3['Infinitos (+inf)'] > 0) |
        (diagnostico_3_3['Infinitos (-inf)'] > 0) |
        (diagnostico_3_3['Total Nulos (NaN)'] > 0)
        ]

    # Veredicto
    if columnas_rotas_3_3.empty:
        print("✅ Veredicto: El dataset 3.3 está LIMPIO.")
        print("➡️ Conclusión: El infinito fue inyectado posteriormente, durante el "
              "paso 3.4 o 3.5.?")
    else:
        print("⚠️ Veredicto: El infinito YA EXISTÍA en el paso 3.3.")
        print("➡️ Conclusión: Revisa el feature engineering en 3.1 o 3.2, o la configuración de tu Scaler en 3.3.")
        display(columnas_rotas_3_3)

except Exception as e:
    print(f"❌ Error al auditar el paso 3.3: {e}")


🔍 Auditando Dataset: TRAIN (Step 3.3 - Transformación)
📁 Ruta: K:\00_Code\Manutenzione\Project_MPPR-AI_B_PHM_America_2024\outputs\runs\regression\phm2024\20260607_133914\phase3_data_preparation\3.3.transformation.transformed_regression_train.parquet
✅ Veredicto: El dataset 3.3 está LIMPIO.
➡️ Conclusión: El infinito fue inyectado posteriormente, durante el paso 3.4 o 3.5.?


In [14]:
#  el .pkl para saber con qué reglas matemáticas se transformó la dataset.

path_x_y_PKL_train_3_3 = base_run_dir / "phase3_data_preparation" / ("3.3"
                                                                     ".transformation.robust_scaler_regression.pkl")

scaler = joblib.load(path_x_y_PKL_train_3_3)

print("Scaler cargado correctamente. Características esperadas:", scaler.n_features_in_)
print("Columnas de entrenamiento:", scaler.feature_names_in_)

(print
 ("----------------------------------------------------------------------------------------"))

# 2. Ver qué aprendió el modelo de tus datos originales
print("Variables transformadas:", scaler.feature_names_in_)
print("Medianas (el centro 0 de tu transformación):", scaler.center_)
print("Escala IQR (lo que usa para comprimir los datos):", scaler.scale_)

Scaler cargado correctamente. Características esperadas: 8
Columnas de entrenamiento: ['oat' 'mgt' 'pa' 'ias' 'np' 'ng' 'trq_margin' 'trq_measured']
----------------------------------------------------------------------------------------
Variables transformadas: ['oat' 'mgt' 'pa' 'ias' 'np' 'ng' 'trq_margin' 'trq_measured']
Medianas (el centro 0 de tu transformación): [ 13.5        589.7        337.8708      88.75        99.76
  94.8          1.83233543  65.57422   ]
Escala IQR (lo que usa para comprimir los datos): [ 11.10928     60.409375   455.2188      58.6723625    4.7975
   7.6794175   14.62694201  18.9       ]


In [7]:
######################################### FORMATTING #########################################

In [29]:

path_x_y_train_3_5_internal = base_run_dir / "phase3_data_preparation" / (
    "3.5.formatting.regression_internal_train.parquet")

query_schema_train = f"DESCRIBE SELECT * FROM parquet_scan('{path_x_y_train_3_5_internal}')"

print("--- ESTRUCTURA DATASET DE x_y_train_3_5_internal ---")
print(duckdb.query(query_schema_train).to_df()[['column_name', 'column_type']])
print("----------------------------------------------------")
query = f"SELECT * FROM '{path_x_y_train_3_5_internal}' LIMIT 5"
df_duck = con.sql(query).df()
display(df_duck)


--- ESTRUCTURA DATASET DE x_y_train_3_5_internal ---
            column_name column_type
0                   oat      DOUBLE
1                   mgt      DOUBLE
2                    pa      DOUBLE
3                   ias      DOUBLE
4                    np      DOUBLE
5                    ng      DOUBLE
6            trq_margin      DOUBLE
7          trq_measured      DOUBLE
8         mgt_oat_ratio      DOUBLE
9  specific_power_index      DOUBLE
----------------------------------------------------


,oat,mgt,pa,ias,np,ng,trq_margin,trq_measured,mgt_oat_ratio,specific_power_index
0,-0.247541,0.276447,0.207231,0.521966,0.052110,0.027346,0.459902,0.742105,2.135963,0.003708
1,-0.913475,0.509855,3.819138,-0.658833,0.006466,-0.026364,-1.436746,-0.028212,2.244107,0.003815
2,-0.765126,-0.759816,0.003013,0.376029,0.043773,-0.494829,0.570673,-0.019800,1.955060,0.003950
3,0.022504,-1.522942,-0.763642,-0.568837,-2.855654,0.660206,0.799622,-1.310805,1.734751,0.003004
4,-0.590005,0.476747,1.282761,0.486848,-0.010224,0.026791,-5.097563,-2.358941,2.208176,0.003747


In [30]:
path_x_y_val_3_5_internal = base_run_dir / "phase3_data_preparation" / ("3.5.formatting"
                                                           ".regression_internal_val.parquet")
query_schema_val = f"DESCRIBE SELECT * FROM parquet_scan('{path_x_y_val_3_5_internal}')"

print("--- ESTRUCTURA DATASET DE VALIDACIÓN ---")
print(duckdb.query(query_schema_val).to_df()[['column_name', 'column_type']])
print("----------------------------------------------------")
query = f"SELECT * FROM '{path_x_y_val_3_5_internal}' LIMIT 5"
df_duck = con.sql(query).df()
display(df_duck)

--- ESTRUCTURA DATASET DE VALIDACIÓN ---
            column_name column_type
0                   oat      DOUBLE
1                   mgt      DOUBLE
2                    pa      DOUBLE
3                   ias      DOUBLE
4                    np      DOUBLE
5                    ng      DOUBLE
6            trq_margin      DOUBLE
7          trq_measured      DOUBLE
8         mgt_oat_ratio      DOUBLE
9  specific_power_index      DOUBLE
----------------------------------------------------


,oat,mgt,pa,ias,np,ng,trq_margin,trq_measured,mgt_oat_ratio,specific_power_index
0,-0.495082,-0.736641,-0.344493,0.078828,0.012507,-0.467483,0.821601,-0.030382,1.939178,0.003893
1,-0.562593,-0.498267,3.387011,-0.955518,0.162585,-0.539103,-1.086369,-0.824033,1.995720,0.003955
2,0.562593,-0.104288,-0.096753,0.102263,-1.640438,0.674530,-0.458732,-0.554192,1.991806,0.003138
3,0.472578,-0.446950,0.465015,-1.013041,-0.087546,-0.563845,-0.136037,-0.749959,1.927715,0.003762
4,0.652608,-0.403911,0.291597,0.013848,0.010422,-0.494829,0.220923,-0.654721,1.923443,0.003732


In [31]:
path_x_y_test_3_5_internal = base_run_dir / "phase3_data_preparation" / ("3.5"
                                                                         ".formatting.regression_internal_test.parquet")
query_schema_val = f"DESCRIBE SELECT * FROM parquet_scan('{path_x_y_test_3_5_internal}')"

print("--- ESTRUCTURA DATASET DE VALIDACIÓN ---")
print(duckdb.query(query_schema_val).to_df()[['column_name', 'column_type']])
print("----------------------------------------------------")
query = f"SELECT * FROM '{path_x_y_test_3_5_internal}' LIMIT 5"
df_duck = con.sql(query).df()
display(df_duck)


--- ESTRUCTURA DATASET DE VALIDACIÓN ---
            column_name column_type
0                   oat      DOUBLE
1                   mgt      DOUBLE
2                    pa      DOUBLE
3                   ias      DOUBLE
4                    np      DOUBLE
5                    ng      DOUBLE
6            trq_margin      DOUBLE
7          trq_measured      DOUBLE
8         mgt_oat_ratio      DOUBLE
9  specific_power_index      DOUBLE
----------------------------------------------------


,oat,mgt,pa,ias,np,ng,trq_margin,trq_measured,mgt_oat_ratio,specific_power_index
0,0.652608,0.895556,1.917977,-0.242874,0.031266,0.191421,-0.365153,0.292369,2.190541,0.003531
1,-1.260208,0.039729,-0.075996,0.319571,-1.496613,0.679739,-1.165242,0.117766,2.171649,0.003395
2,-0.472578,-0.160571,-0.606294,-0.937409,0.006253,-0.199234,0.667504,0.567502,2.061123,0.003802
3,0.517585,0.996534,0.446267,0.442074,-0.760813,0.684948,-0.839755,0.255332,2.222640,0.003285
4,0.202533,-0.663804,2.221963,-0.563511,0.083377,-0.610723,-0.184499,-0.951017,1.902388,0.003847


In [32]:
query_conteos = f"""
SELECT 'x_y_train_3_5_internal' as dataset, count(*) as total_registros FROM parquet_scan('{path_x_y_train_3_5_internal}')
UNION ALL
SELECT 'x_y_val_3_5_internal' as dataset, count(*) as total_registros FROM parquet_scan('{path_x_y_val_3_5_internal}')
UNION ALL
SELECT 'x_y_test_3_5_internal' as dataset, count(*) as total_registros FROM parquet_scan('{path_x_y_test_3_5_internal}')
"""

print(duckdb.query(query_conteos).to_df())

                  dataset  total_registros
0  x_y_train_3_5_internal             4895
1    x_y_val_3_5_internal             1049
2   x_y_test_3_5_internal             1050


In [33]:

# 1. Agrupamos los paths en un diccionario
datasets_a_auditar = {
    "TRAIN": path_x_y_train_3_5_internal,
    "VALIDATION": path_x_y_val_3_5_internal,
    "TEST": path_x_y_test_3_5_internal
}

# 2. Iteramos sobre cada dataset
for nombre, ruta in datasets_a_auditar.items():
    print(f"\n{'='*60}")
    print(f"🔍 Auditando Dataset: {nombre}")
    print(f"📁 Ruta: {ruta}")
    print(f"{'='*60}")

    try:
        # Cargar el dataset usando DuckDB
        query = f"SELECT * FROM parquet_scan('{ruta}')"
        df = duckdb.query(query).to_df()

        # Aislar solo las columnas numéricas (para evitar errores con strings o fechas)
        df_num = df.select_dtypes(include=[np.number])

        # Construir el reporte de diagnóstico
        diagnostico = pd.DataFrame({
            'Total Nulos (NaN)': df_num.isna().sum(),
            'Infinitos (+inf)': np.isposinf(df_num).sum(),
            'Infinitos (-inf)': np.isneginf(df_num).sum(),
            'Valor Máximo': df_num.max(),
            'Valor Mínimo': df_num.min()
        })

        # Filtrar para mostrar SOLO las columnas que tienen problemas
        columnas_rotas = diagnostico[
            (diagnostico['Infinitos (+inf)'] > 0) |
            (diagnostico['Infinitos (-inf)'] > 0) |
            (diagnostico['Total Nulos (NaN)'] > 0)
            ]

        # Imprimir resultados
        if columnas_rotas.empty:
            print(f"✅ ¡Todo en orden! El dataset {nombre} está limpio (0 nulos, 0 infinitos).\n")
        else:
            print(f"⚠️ ¡ALERTA! Se encontraron columnas con problemas en {nombre}:")
            display(columnas_rotas)

    except Exception as e:
        print(f"❌ Error al intentar leer o procesar el dataset {nombre}: {e}\n")


🔍 Auditando Dataset: TRAIN
📁 Ruta: K:\00_Code\Manutenzione\Project_MPPR-AI_B_PHM_America_2024\outputs\runs\regression\phm2024\20260607_133914\phase3_data_preparation\3.5.formatting.regression_internal_train.parquet
✅ ¡Todo en orden! El dataset TRAIN está limpio (0 nulos, 0 infinitos).


🔍 Auditando Dataset: VALIDATION
📁 Ruta: K:\00_Code\Manutenzione\Project_MPPR-AI_B_PHM_America_2024\outputs\runs\regression\phm2024\20260607_133914\phase3_data_preparation\3.5.formatting.regression_internal_val.parquet
✅ ¡Todo en orden! El dataset VALIDATION está limpio (0 nulos, 0 infinitos).


🔍 Auditando Dataset: TEST
📁 Ruta: K:\00_Code\Manutenzione\Project_MPPR-AI_B_PHM_America_2024\outputs\runs\regression\phm2024\20260607_133914\phase3_data_preparation\3.5.formatting.regression_internal_test.parquet
✅ ¡Todo en orden! El dataset TEST está limpio (0 nulos, 0 infinitos).

